In [54]:
import pandas as pd

In [55]:
items=pd.read_csv("C:/Users/Supradha/ E-Commerce User Funnel & Drop-off Analysis/dataset/olist_order_items_dataset.csv")

In [56]:
items

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14
...,...,...,...,...,...,...,...
112645,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.99,43.41
112646,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53
112647,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,99.90,16.95
112648,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,2017-08-21 00:04:32,55.99,8.72


In [57]:
items.shape

(112650, 7)

In [58]:
items.dtypes

order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object

In [59]:
items.isnull().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [60]:
# duplicate rows
items.duplicated().sum()

0

In [61]:
# Remove full duplicate rows 
items.drop_duplicates(inplace=True)

In [62]:
# Parse shipping_limit_date ───────────────────────────
items['shipping_limit_date'] = pd.to_datetime(items['shipping_limit_date'], errors='coerce')

In [63]:
# Validate numeric columns
print(items['price'].describe())

count    112650.000000
mean        120.653739
std         183.633928
min           0.850000
25%          39.900000
50%          74.990000
75%         134.900000
max        6735.000000
Name: price, dtype: float64


In [64]:
# validate numeric columns
print(items['freight_value'].describe())

count    112650.000000
mean         19.990320
std          15.806405
min           0.000000
25%          13.080000
50%          16.260000
75%          21.150000
max         409.680000
Name: freight_value, dtype: float64


In [65]:
# Flag negative or zero prices (data errors)
invalid_price = items[items['price'] <= 0]
print(f"\n  Rows with price <= 0 : {len(invalid_price)}")


  Rows with price <= 0 : 0


In [66]:
items = items[items['price'] > 0]   # Remove — cannot be real

In [67]:
invalid_freight = items[items['freight_value'] < 0]
print(f"  Rows with freight < 0 : {len(invalid_freight)}")
items = items[items['freight_value'] >= 0]

  Rows with freight < 0 : 0


In [68]:
# Outlier check on price
Q1 = df['price'].quantile(0.25)
Q3 = df['price'].quantile(0.75)
IQR = Q3 - Q1
upper_fence = Q3 + 3 * IQR    # 3×IQR = extreme outliers only
outliers = df[df['price'] > upper_fence]
print(f"\n  Price extreme outliers (>3×IQR fence {upper_fence:.2f}) : " f"{len(outliers)}")


  Price extreme outliers (>3×IQR fence 419.90) : 4074


In [69]:
# Do NOT delete — high-value items are valid. Just flag them.
items['is_price_outlier'] = items['price'] > upper_fence

In [70]:
# Derived columns 
items['total_item_value'] = items['price'] + items['freight_value']

In [71]:
items.shape

(112650, 9)

In [72]:
items.to_csv("cleaned_items.csv",index=False)